In [1]:
import numpy as np
import pandas as pd

from datetime import datetime, timezone, timedelta


In [65]:
import hax
pax_version = '6.10.1'

hax.init(experiment='XENON1T',
         pax_version_policy=pax_version,
         minitree_paths=['scratch-midway2/miniforest/pax_v'+ pax_version,
                         '/project2/lgrandi/xenon1t/minitrees/pax_v'+ pax_version,
                         '/dali/lgrandi/xenon1t/minitrees/pax_v'+ pax_version,
                        ],
         main_data_paths=['/dali/lgrandi/xenon1t/processed/pax_v'+ pax_version,
                          '/project2/lgrandi/xenon1t/processed/pax_v'+ pax_version,
                         ],)

dsets = hax.runs.datasets

dsets = hax.runs.tags_selection(dsets, include=['sciencerun0','sciencerun1','GW'],
                            exclude=['NoMV','Flash','PMTramping','readoutbug','flash','bad', 'messy', '*trip', '*quake','test','NG','MVoff'])

dsets = dsets[
    (dsets['location'] != "")
    & (dsets.source__type == 'none')
]



In [66]:
dataset_index = []
dataset_start = []
dataset_end = []
for i in range(0, np.shape(dsets)[0]):
    id = dsets.index[i]
    dataset_index.append(i)
    dataset_start.append(dsets.start[id].replace(tzinfo=timezone.utc).timestamp())
    dataset_end.append(dsets.end[id].replace(tzinfo=timezone.utc).timestamp())


In [67]:
# intialise data of lists.

from datetime import datetime, timezone, timedelta

GW_8_23_datetime = datetime(2017, 8, 23, 13, 13, 58, tzinfo=timezone.utc)
GW_8_18_datetime = datetime(2017, 8, 18, 2, 25, 9, tzinfo=timezone.utc)
GW_8_17_datetime = datetime(2017, 8, 17, 12, 41, 4, tzinfo=timezone.utc)
GW_7_29_datetime = datetime(2017, 7, 29, 18, 56, 29, tzinfo=timezone.utc)
GW_1_04_datetime = datetime(2017, 1, 4, 10, 11, 58, tzinfo=timezone.utc)

GW_1_04_time = GW_1_04_datetime.timestamp()
GW_7_29_time = GW_7_29_datetime.timestamp()
GW_8_17_time = GW_8_17_datetime.timestamp()
GW_8_18_time = GW_8_18_datetime.timestamp()
GW_8_23_time = GW_8_23_datetime.timestamp()

GW = ['GW170104', 'GW170729', 'GW170817', 'GW170818', 'GW170823']

data = {'Event': GW,
        'Event_time': [GW_1_04_time, GW_7_29_time, GW_8_17_time, GW_8_18_time, GW_8_23_time],
        }
Data = pd.DataFrame(data)

Data

,Event,Event_time
0,GW170104,1.483525e+09
1,GW170729,1.501355e+09
2,GW170817,1.502974e+09
3,GW170818,1.503023e+09
4,GW170823,1.503494e+09


In [68]:
Data['Time before'] = ''
Data['Time after'] = ''
Data['dsets_index'] = ''

for j in range(0, np.shape(Data['Event_time'])[0]):
    for i in range(0, np.size(dataset_start)):
        if Data['Event_time'][j] >= dataset_start[i] and Data['Event_time'][j] <= dataset_end[i]:
            Data['Time before'][j] = Data['Event_time'][j] - dataset_start[i] 
            Data['Time after'][j] = dataset_end[i] - Data['Event_time'][j]
            Data['dsets_index'][j] = dataset_index[i]



/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing

In [69]:
Data

,Event,Event_time,Time before,Time after,dsets_index
0,GW170104,1.483525e+09,2336,1268,595
1,GW170729,1.501355e+09,2272,1332,3481
2,GW170817,1.502974e+09,1982,1622,3540
3,GW170818,1.503023e+09,848,2756,3554
4,GW170823,1.503494e+09,1653,1951,3673


Datasets are completely within 500 seconds of GW

In [80]:
def extract_data(datasets):
    exclude_runs = []
    #exclude_runs = [16087, 15703, 15798, 16020, 15722, 15954, 16188]
    datasets = datasets.query('not number in @exclude_runs')
    data = hax.minitrees.load(datasets.number.values, [#"Basics",
                                                "Fundamentals",
                                                "Proximity",
                                                #"TotalProperties",
                                                "FlashIdentification",
                                                "TailCut",
                                               ], 
                          #preselection=["abs(nearest_muon_veto_trigger)<2e6"],
    #                           num_workers=1,
                          #force_reload=True
                         )
    return data

In [73]:
#comment out TailCut before running this
GW170104_data = extract_data(dsets[dsets.index == dsets.index[595]]) 

In [81]:
GW170729_data = extract_data(dsets[dsets.index == dsets.index[3481]])
GW170817_data = extract_data(dsets[dsets.index == dsets.index[3540]])
GW170818_data = extract_data(dsets[dsets.index == dsets.index[3554]])
GW170823_data = extract_data(dsets[dsets.index == dsets.index[3673]])

## DAQVeto

Adopted from https://github.com/XENON1T/lax/blob/v1.0.0/lax/lichens/sciencerun0.py#L103

### EndOfRunCheck
Removes the last 21 seconds of each run.
From above table we can see that it is not an issue.

### HEVCheck

It is used only in calibration modes

In [36]:
def HEVCheck(df):
    for i in range (0, np.shape(df)[0]):
        if (abs(df['nearest_hev'][i]) < df['event_duration'][i] / 2):
            print("problem")
            

In [37]:
HEVCheck(GW170104_data)

In [30]:
HEVCheck(GW170729_data)

In [31]:
HEVCheck(GW170817_data)

In [32]:
HEVCheck(GW170818_data)

In [33]:
HEVCheck(GW170823_data)

### BusyCheck

In [12]:
def BusyCheck(df, GW_time):
    deadtime = 0
    for i in range (0, np.shape(df)[0]):
        if (abs(df['nearest_busy'][i]) < df['event_duration'][i] / 2):
            if (df['event_time'][i] * 1e-9 - GW_time < 500 and df['event_time'][i] * 1e-9 - GW_time > -500):
                deadtime = deadtime + df['event_duration'][i] * 1e-9 #in seconds     
    return deadtime    
            

In [75]:
Data['DAQ_deadtime'] = ''
Data['DAQ_deadtime'][0] = BusyCheck(GW170104_data, Data['Event_time'][0])
Data['DAQ_deadtime'][1] = BusyCheck(GW170729_data, Data['Event_time'][1])
Data['DAQ_deadtime'][2] = BusyCheck(GW170817_data, Data['Event_time'][2])
Data['DAQ_deadtime'][3] = BusyCheck(GW170818_data, Data['Event_time'][3])
Data['DAQ_deadtime'][4] = BusyCheck(GW170823_data, Data['Event_time'][4])
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  from ipykernel import kernelapp as app
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  app.launch_new_instance()
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation

,Event,Event_time,Time before,Time after,dsets_index,DAQ_deadtime
0,GW170104,1.483525e+09,2336,1268,595,0.0940288
1,GW170729,1.501355e+09,2272,1332,3481,0.206651
2,GW170817,1.502974e+09,1982,1622,3540,0.0510577
3,GW170818,1.503023e+09,848,2756,3554,0.0907836
4,GW170823,1.503494e+09,1653,1951,3673,0.0357239


### BusyTypeCheck

In [14]:
def BusyTypeCheck(df, GW_time):
    deadtime = 0
    for i in range (0, np.shape(df)[0]):
        if (df['previous_busy_off'][i] > df['previous_busy_on'][i]):
            if (df['event_time'][i] * 1e-9 - GW_time < 500 and df['event_time'][i] * 1e-9 - GW_time > -500):
                deadtime = deadtime + (df['event_duration'][i]) * 1e-9 #in seconds    
                '''
                if (df['previous_busy_on'][i] < df['event_duration'][i]/2):
                    deadtime = deadtime + (df['event_duration'][i]) * 1e-9 #in seconds    
                else:
                    deadtime = deadtime + (df['event_duration'][i]/2 + df['previous_busy_on'][i]) * 1e-9 #in seconds    
                '''
    return deadtime    

In [76]:
Data['DAQ_deadtime'][0] = Data['DAQ_deadtime'][0] + BusyTypeCheck(GW170104_data, Data['Event_time'][0])
Data['DAQ_deadtime'][1] = Data['DAQ_deadtime'][1] + BusyTypeCheck(GW170729_data, Data['Event_time'][1])
Data['DAQ_deadtime'][2] = Data['DAQ_deadtime'][2] + BusyTypeCheck(GW170817_data, Data['Event_time'][2])
Data['DAQ_deadtime'][3] = Data['DAQ_deadtime'][3] + BusyTypeCheck(GW170818_data, Data['Event_time'][3])
Data['DAQ_deadtime'][4] = Data['DAQ_deadtime'][4] + BusyTypeCheck(GW170823_data, Data['Event_time'][4])
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  if __name__ == '__main__':
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  from ipykernel import kernelapp as app
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentatio

,Event,Event_time,Time before,Time after,dsets_index,DAQ_deadtime
0,GW170104,1.483525e+09,2336,1268,595,0.100029
1,GW170729,1.501355e+09,2272,1332,3481,0.346638
2,GW170817,1.502974e+09,1982,1622,3540,0.0510577
3,GW170818,1.503023e+09,848,2756,3554,0.0980093
4,GW170823,1.503494e+09,1653,1951,3673,0.0357239


## Muon Veto

Adopted from https://github.com/XENON1T/lax/blob/master/lax/lichens/sciencerun0.py#L896

In [25]:
def MVnotWorking(df):
    for i in range (0, np.shape(df)[0]):
        if (df['nearest_muon_veto_trigger'][i] < -2e10 and df['nearest_muon_veto_trigger'][i] > 2e10):
            print("problem")

In [26]:
MVnotWorking(GW170823_data)

In [93]:
MVnotWorking(GW170818_data)

In [95]:
MVnotWorking(GW170817_data)

In [96]:
MVnotWorking(GW170729_data)

In [97]:
MVnotWorking(GW170104_data)

In [16]:
def MounVeto(df, GW_time):
    deadtime = 0
    for i in range (0, np.shape(df)[0]):
        if (df['nearest_muon_veto_trigger'][i] > -2e6 and df['nearest_muon_veto_trigger'][i] < 3e6):
                if (df['event_time'][i] * 1e-9 - GW_time < 500 and df['event_time'][i] * 1e-9 - GW_time > -500):
                    deadtime = deadtime + (df['event_duration'][i]) * 1e-9 #in seconds  
    return deadtime

In [77]:
Data['muon_deadtime'] = ''
Data['muon_deadtime'][0] = MounVeto(GW170104_data, Data['Event_time'][0])
Data['muon_deadtime'][1] = MounVeto(GW170729_data, Data['Event_time'][1])
Data['muon_deadtime'][2] = MounVeto(GW170817_data, Data['Event_time'][2])
Data['muon_deadtime'][3] = MounVeto(GW170818_data, Data['Event_time'][3])
Data['muon_deadtime'][4] = MounVeto(GW170823_data, Data['Event_time'][4])
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  from ipykernel import kernelapp as app
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  app.launch_new_instance()
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation

,Event,Event_time,Time before,Time after,dsets_index,DAQ_deadtime,muon_deadtime
0,GW170104,1.483525e+09,2336,1268,595,0.100029,0.024387
1,GW170729,1.501355e+09,2272,1332,3481,0.346638,0.0311426
2,GW170817,1.502974e+09,1982,1622,3540,0.0510577,0.0192684
3,GW170818,1.503023e+09,848,2756,3554,0.0980093,0.0306173
4,GW170823,1.503494e+09,1653,1951,3673,0.0357239,0.0365527


## Flash

Any event within a flash?

In [92]:
def InFlash(df):
    for i in range (0, np.shape(df)[0]):
        if (df['inside_flash'][i] == True):
            print("problem")


In [94]:
InFlash(GW170104_data)
InFlash(GW170729_data)
InFlash(GW170817_data)
InFlash(GW170818_data)
InFlash(GW170823_data)

Any event close to the flash?

In [95]:
def nearFlash(df):
    for i in range (0, np.shape(df)[0]):
        if (df['nearest_flash'][i] < 120e9):
            print("problem")

In [108]:
nearFlash(GW170104_data)
nearFlash(GW170729_data)
nearFlash(GW170817_data)
nearFlash(GW170818_data)
nearFlash(GW170823_data)

Any event just before the flash?

In [100]:
def beforeFlash(df):
    for i in range (0, np.shape(df)[0]):
        if (df['nearest_flash'][i] < (-10e9 - df['flashing_width'][i] * 1e9)):
            print("problem")

In [109]:
beforeFlash(GW170104_data)
beforeFlash(GW170729_data)
beforeFlash(GW170817_data)
beforeFlash(GW170818_data)
beforeFlash(GW170823_data)

In [107]:
GW170729_data['nearest_flash']

0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
5       NaN
6       NaN
7       NaN
8       NaN
9       NaN
10      NaN
11      NaN
12      NaN
13      NaN
14      NaN
15      NaN
16      NaN
17      NaN
18      NaN
19      NaN
20      NaN
21      NaN
22      NaN
23      NaN
24      NaN
25      NaN
26      NaN
27      NaN
28      NaN
29      NaN
         ..
19190   NaN
19191   NaN
19192   NaN
19193   NaN
19194   NaN
19195   NaN
19196   NaN
19197   NaN
19198   NaN
19199   NaN
19200   NaN
19201   NaN
19202   NaN
19203   NaN
19204   NaN
19205   NaN
19206   NaN
19207   NaN
19208   NaN
19209   NaN
19210   NaN
19211   NaN
19212   NaN
19213   NaN
19214   NaN
19215   NaN
19216   NaN
19217   NaN
19218   NaN
19219   NaN
Name: nearest_flash, Length: 19220, dtype: float64

## s2Tails

In [ ]:
'''
#https://github.com/XENON1T/hax/blob/master/hax/treemakers/trigger.py

look_back = 50

s2 = data['s2_0_area'].values
s2[np.isnan(s2)] = 0
t = data['event_time'].values

s2_over_tdiff_lookback = np.zeros((len(t), look_back + 1))

for i in range(1, look_back + 1):
            s2_over_tdiff_lookback[i:, i] = s2[:-i]/(t[i:] - t[:-i])
s2_over_tdiff = s2_over_tdiff_lookback.max(axis=1)
'''

In [83]:
def s2Tails(df, GW_time):
    deadtime = 0
    for i in range (0, np.shape(df)[0]):
        if((df['s2_over_tdiff'][i] > 0.04)):
            if (df['event_time'][i] * 1e-9 - GW_time < 500 and df['event_time'][i] * 1e-9 - GW_time > -500):
                deadtime = deadtime + (df['event_duration'][i]) * 1e-9 #in seconds  
    return deadtime

In [89]:
Data['s2Tail_deadtime'] = ''
Data['s2Tail_deadtime'][0] = 0
Data['s2Tail_deadtime'][1] = s2Tails(GW170729_data, Data['Event_time'][1])
Data['s2Tail_deadtime'][2] = s2Tails(GW170817_data, Data['Event_time'][2])
Data['s2Tail_deadtime'][3] = s2Tails(GW170818_data, Data['Event_time'][3])
Data['s2Tail_deadtime'][4] = s2Tails(GW170823_data, Data['Event_time'][4])
Data

/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  from ipykernel import kernelapp as app
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  app.launch_new_instance()
/cvmfs/xenon.opensciencegrid.org/releases/anaconda/2.4/envs/pax_v6.10.1/lib/python3.4/site-packages/ipykernel/__main__.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation

,Event,Event_time,Time before,Time after,dsets_index,DAQ_deadtime,muon_deadtime,s2Tail_deadtime
0,GW170104,1.483525e+09,2336,1268,595,0.100029,0.024387,0
1,GW170729,1.501355e+09,2272,1332,3481,0.346638,0.0311426,0.689852
2,GW170817,1.502974e+09,1982,1622,3540,0.0510577,0.0192684,0.599012
3,GW170818,1.503023e+09,848,2756,3554,0.0980093,0.0306173,0.63471
4,GW170823,1.503494e+09,1653,1951,3673,0.0357239,0.0365527,0.62894


# s2only

In [19]:
Data['s2only deadtime'] = Data['muon_deadtime'] + Data['DAQ_deadtime'] + 2 
#2 seconds for Lone-S1 signal

In [18]:
Data['s2only livetime'] = 1000 - Data['s2only deadtime']
Data

,Event,Event_time,Time before,Time after,dsets_index,DAQ_deadtime,muon_deadtime,s2only deadtime,s2only livetime
0,GW170104,1.483525e+09,2336,1268,595,0.100029,0.024387,2.12442,997.876
1,GW170729,1.501355e+09,2272,1332,3481,0.346638,0.0311426,2.37778,997.622
2,GW170817,1.502974e+09,1982,1622,3540,0.0510577,0.0192684,2.07033,997.93
3,GW170818,1.503023e+09,848,2756,3554,0.0980093,0.0306173,2.12863,997.871
4,GW170823,1.503494e+09,1653,1951,3673,0.0357239,0.0365527,2.07228,997.928


# Nuclear Recoil

In [91]:
Data['nuclear recoil deadtime'] = Data['DAQ_deadtime'] + Data['muon_deadtime'] + Data['s2Tail_deadtime']
Data['nuclear recoil livetime'] = 1000 - Data['nuclear recoil deadtime']
Data

,Event,Event_time,Time before,Time after,dsets_index,DAQ_deadtime,muon_deadtime,s2Tail_deadtime,nuclear recoil deadtime,nuclear recoil livetime
0,GW170104,1.483525e+09,2336,1268,595,0.100029,0.024387,0,0.124416,999.876
1,GW170729,1.501355e+09,2272,1332,3481,0.346638,0.0311426,0.689852,1.06763,998.932
2,GW170817,1.502974e+09,1982,1622,3540,0.0510577,0.0192684,0.599012,0.669338,999.331
3,GW170818,1.503023e+09,848,2756,3554,0.0980093,0.0306173,0.63471,0.763337,999.237
4,GW170823,1.503494e+09,1653,1951,3673,0.0357239,0.0365527,0.62894,0.701216,999.299
